In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [ ]:
# Load the dataset
data = pd.read_csv('Churn_Modelling.csv')

# Drop unnecessary identifier columns that do not help in prediction
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)



label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')

# Fit & transform Geography column into binary one-hot columns
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

# Create a new DataFrame with proper column names
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Replace original Geography column with encoded columns
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

# Train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



scaler = StandardScaler()

# Fit scaler on training data and transform it
X_train = scaler.fit_transform(X_train)

# Only transform test data (do NOT fit again)
X_test = scaler.transform(X_test)



with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


In [ ]:


## Define a function to create the ANN model (used by KerasClassifier for hyperparameter tuning)
def create_model(neurons=32, layers=1):
    # Initialize a sequential model
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss="binary_crossentropy", metrics=['accuracy'])

    # Return the compiled model (required by KerasClassifier)
    return model


In [ ]:

model = KerasClassifier(layers=1, neurons=32, build_fn=create_model, verbose=1)


In [ ]:


param_grid = {
    'neurons': [16, 32, 64, 128],  # possible neurons per layer
    'layers': [1, 2],               # possible number of hidden layers
    'epochs': [50, 100]             # possible number of epochs
}


In [6]:
# this will take time like 5 6 min
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


e:\generative AI\annclassification\annclassification\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)




Epoch 1/100


250/250 [==============================] - 1s 871us/step - loss: 0.5661 - accuracy: 0.7185
Epoch 2/100
250/250 [==============================] - 0s 914us/step - loss: 0.4410 - accuracy: 0.8163
Epoch 3/100
250/250 [==============================] - 0s 804us/step - loss: 0.4189 - accuracy: 0.8234
Epoch 4/100
250/250 [==============================] - 0s 870us/step - loss: 0.4043 - accuracy: 0.8288
Epoch 5/100
250/250 [==============================] - 0s 937us/step - loss: 0.3920 - accuracy: 0.8379
Epoch 6/100
250/250 [==============================] - 0s 870us/step - loss: 0.3813 - accuracy: 0.8410
Epoch 7/100
250/250 [==============================] - 0s 934us/step - loss: 0.3729 - accuracy: 0.8474
Epoch 8/100
250/250 [==============================] - 0s 870us/step - loss: 0.3665 - accuracy: 0.8493
Epoch 9/100
250/250 [==============================] - 0s 938us/step - loss: 0.3614 - accuracy: 0.8526
Epoch 10/100
250/250 [==============================] - 0s 869us/step